# TP Máquina de Turing Universal
---

**¿Por qué se dice que una MTU es capaz de simular cualquier Máquina de Turing?**

**Suponer que se tiene una MT M que acepta todas las cadenas que terminan en 01. Indicar qué debería hacer una MTU con las siguientes entradas. Explicar en cada caso si la MTU acepta o rechaza y por qué**
* U(⟨M⟩,1101)
* U(⟨M⟩,100)
* U(⟨M⟩,01)
* U(⟨M⟩,111)

**Dada la siguiente MT M:** 

| Q   | 0        | 1        |
| --  | --       | --       |
| q0  | (q1,1,R) | (q1,0,R) |
| q1  | (qf,0,R) | (qf,1,R) |
| qf  | -        | -        |

1. Explicar que hace M 
2. Explicar qué información debería recibir una MTU para poder simular M
3. Codificar la cintar de MTU sabiendo que configuración de la cinta de MT M es 1 q0 0 1 1

1. **Codificación de una máquina simple**  
   * Definir una máquina $M$ que ...
   * Codificar sus estados, símbolos y transiciones en forma numérica

![alt text](image-1.png) ![alt text](image.png)
| Estado     | Función                     | Codificación |
| ---------- | --------------------------- | -----------: |
| `q0`       | Buscar un 0                 |        `000` |
| `q1`       | Buscar un 1                 |        `001` |
| `q2`       | Volver al comienzo          |        `010` |
| `q3`       | Verificar que solo queden Y |        `011` |
| `q_accept` | Aceptar                     |        `100` |
| `q_reject` | Rechazar                    |        `101` |

| Símbolo | Binario |       
| ------- | ------: |
| B       |     000 |
| 0       |     001 |
| 1       |     010 |
| X       |     011 |
| Y       |     100 |

| Movimiento | Binario |
| ---------- | ------: |
| L          |      00 |
| R          |      01 |
| S          |      10 |

| Transición original    | Codificación binaria |     |     |     |     |
| ---------------------- | -------------------- | --- | --- | --- | --- |
| δ(q0,X)=(q0,X,R)       | `000                 | 011 | 000 | 011 | 01` |
| δ(q0,0)=(q1,X,R)       | `000                 | 001 | 001 | 011 | 01` |
| δ(q0,Y)=(q3,Y,R)       | `000                 | 100 | 011 | 100 | 01` |
| δ(q0,1)=(q_R,1,S) | `000                 | 010 | 101 | 010 | 10` |
| δ(q0,B)=(q_A,B,S) | `000                 | 000 | 100 | 000 | 10` |
| δ(q1,0)=(q1,0,R)       | `001                 | 001 | 001 | 001 | 01` |
| δ(q1,X)=(q1,X,R)       | `001                 | 011 | 001 | 011 | 01` |
| δ(q1,Y)=(q1,Y,R)       | `001                 | 100 | 001 | 100 | 01` |
| δ(q1,1)=(q2,Y,L)       | `001                 | 010 | 010 | 100 | 00` |
| δ(q1,B)=(q_R,B,S) | `001                 | 000 | 101 | 000 | 10` |
| δ(q2,0)=(q2,0,L)       | `010                 | 001 | 010 | 001 | 00` |
| δ(q2,1)=(q2,1,L)       | `010                 | 010 | 010 | 010 | 00` |
| δ(q2,X)=(q2,X,L)       | `010                 | 011 | 010 | 011 | 00` |
| δ(q2,Y)=(q2,Y,L)       | `010                 | 100 | 010 | 100 | 00` |
| δ(q2,B)=(q0,B,R)       | `010                 | 000 | 000 | 000 | 01` |
| δ(q3,Y)=(q3,Y,R)       | `011                 | 100 | 011 | 100 | 01` |
| δ(q3,B)=(q_A,B,S)      | `011              | 000 | 100 | 000 | 10` |
| δ(q3,0)=(q_R,0,S) | `011                   | 001 | 101 | 001 | 10` |
| δ(q3,1)=(q_R,1,S) | `011                   | 010 | 101 | 010 | 10` |





2. **Simulación básica**  
   * Implementar en Python un programa que reciba:
     * La codificación de una máquina $M$
     * Una cadena de entrada $w$
   * El programa debe simular paso a paso la ejecución de $M$ sobre $w$

In [ ]:
# ============================================================
# MAQUINA DE TURING UNIVERSAL 
# ============================================================


# ------------------------------------------------------------
# Codificación de símbolos
# ------------------------------------------------------------

SIMBOLOS = {
    "B": 000,     # Blanco
    "0": 001,
    "1": 010,
    "X": 011,
    "Y": 100    
}
SIMBOLOS_INV = {
    000: "B",
    001: "0",
    010: "1",
    011: "X",
    100: "Y"
}


# ------------------------------------------------------------
# Codificación de movimientos
# ------------------------------------------------------------
MOVIMIENTOS = {
    00: "L",
    01: "R",
    10: "S"
}
# ------------------------------------------------------------
# Función para buscar una transición
# ------------------------------------------------------------

def buscar_transicion(maquina, estado, simbolo):

    for transicion in maquina:

        estado_actual = transicion[0]
        simbolo_leido = transicion[1]

        if estado_actual == estado and simbolo_leido == simbolo:
            return transicion

    return None
# ------------------------------------------------------------
# Función que muestra la cinta
# ------------------------------------------------------------

def mostrar_cinta(cinta, cabezal, estado):

    minimo = min(cinta.keys())
    maximo = max(cinta.keys())

    print(f"Estado: q{estado}")

    for posicion in range(minimo, maximo + 1):

        simbolo = SIMBOLOS_INV[cinta.get(posicion, 0)]

        if posicion == cabezal:
            print(f"[{simbolo}]", end=" ")
        else:
            print(f" {simbolo} ", end=" ")

    print()
    print()

# ------------------------------------------------------------
# Máquina de Turing Universal
# ------------------------------------------------------------

def mtu(maquina, palabra, max_pasos=1000):

    # Cinta inicialmente vacía
    cinta = {}

    # Guardamos la palabra de entrada
    for i, simbolo in enumerate(palabra):

        if simbolo not in SIMBOLOS:
            print("Símbolo inválido.")
            return False

        cinta[i] = SIMBOLOS[simbolo]

    # Si alguna posición no existe se considera blanco
    cinta.setdefault(0, 0)

    # Estado inicial
    estado = 0

    # Estados finales
    estado_aceptacion = 4
    estado_rechazo = 5

    # Cabezal comienza en la primera posición
    cabezal = 0

    pasos = 0

    print("=" * 50)
    print("INICIO DE LA SIMULACION")
    print("=" * 50)

    mostrar_cinta(cinta, cabezal, estado)

    while pasos < max_pasos:

        # Comprobar estados finales

        if estado == estado_aceptacion:

            print("=" * 50)
            print("CADENA ACEPTADA")
            print("=" * 50)

            return True

        if estado == estado_rechazo:

            print("=" * 50)
            print("CADENA RECHAZADA")
            print("=" * 50)

            return False

        # Obtengo símbolo debajo del cabezal
        simbolo_actual = cinta.get(cabezal, 0)

        # Busco transición
        transicion = buscar_transicion(
            maquina,
            estado,
            simbolo_actual
        )

        if transicion is None:

            print("No existe una transición válida.")
            print("La máquina se detiene.")

            return False

        # Descomponer transición
        estado_actual = transicion[0]
        simbolo_leido = transicion[1]
        nuevo_estado = transicion[2]
        simbolo_escrito = transicion[3]
        movimiento = transicion[4]

        print(f"Paso {pasos + 1}")

        print(
            f"δ(q{estado_actual}, "
            f"{SIMBOLOS_INV[simbolo_leido]}) = "
            f"(q{nuevo_estado}, "
            f"{SIMBOLOS_INV[simbolo_escrito]}, "
            f"{MOVIMIENTOS[movimiento]})"
        )

        # Escribir nuevo símbolo
        cinta[cabezal] = simbolo_escrito

        # Mover cabezal
        if movimiento == 0:

            cabezal -= 1

        elif movimiento == 1:

            cabezal += 1

        # movimiento 2 = no mover

        # Cambiar estado
        estado = nuevo_estado

        # Agregar blanco automáticamente
        cinta.setdefault(cabezal, 0)

        pasos += 1

        mostrar_cinta(cinta, cabezal, estado)

    print("Se alcanzó el máximo de pasos.")

    return False

3. **Pruebas de funcionamiento**  
   * Probar la simulación con diferentes entradas
   * Documentar los resultados

4. **Informe final**  
   * Explicar la codificación utilizada
   * Mostrar ejemplos de ejecución
   * Reflexionar sobre la relación entre la MTU y las computadoras modernas